# FFT Evaluation of ICA

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We compare the frequency spectrum of channel P4 before and after ICA cleaning (excluding the highest-variance component) to evaluate the effect of artifact removal on brain wave bands.

## Expected outputs

- Spectrum before cleaning on top
- Spectrum after cleaning on bottom
- Five brain wave bands shaded in both

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| n_components | 4 | ICA components |
| exclude | highest variance | Excluded component |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply ICA and compare spectra

We apply ICA, exclude the highest-variance component, then compare the spectrum before and after cleaning.


In [ ]:
import mne
from scipy.fft import fft, fftfreq

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

# NOTE: ICA works best with more channels than components.
ica = mne.preprocessing.ICA(
    n_components=3, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)

component_variances = np.var(ica.get_sources(raw).get_data(), axis=1)
exclude_idx = int(np.argmax(component_variances))
ica.exclude = [exclude_idx]
cleaned_raw = ica.apply(raw.copy(), verbose=False)
cleaned_data = cleaned_raw.get_data()[0] * 1e6
original_data = eeg_data[:, 0]

def compute_spectrum(data, fs):
    spectrum = fft(data)
    freqs = fftfreq(len(data), 1 / fs)
    magnitude = np.abs(spectrum)
    pos_mask = freqs >= 0
    return freqs[pos_mask], magnitude[pos_mask]

freqs_orig, mag_orig = compute_spectrum(original_data, fs)
freqs_clean, mag_clean = compute_spectrum(cleaned_data, fs)
print(f'Excluded component: IC{exclude_idx}')


## 5. Interactive plot

**What to look for:**

- The difference between spectra shows the cleaning effect
- Shaded bands represent the five brain waves
- Zoom in to inspect specific frequency ranges



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('FFT Before ICA - Channel P4',
                                    'FFT After ICA - Channel P4'))
fig.add_trace(go.Scatter(x=freqs_orig, y=mag_orig, name='Before',
                         line=dict(color='black', width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs_clean, y=mag_clean, name='After',
                         line=dict(color='green', width=0.8)), row=2, col=1)
for name, fmin, fmax, color in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=1, col=1)
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=2, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_xaxes(range=[0, 80], row=2, col=1)
fig.update_layout(height=700, title_text='FFT Evaluation - Before vs After ICA',
                  xaxis_title='Frequency (Hz)', xaxis2_title='Frequency (Hz)',
                  yaxis_title='Magnitude', yaxis2_title='Magnitude',
                  showlegend=False)
fig.show()


## What did we learn?

- Spectrum comparison reveals the cleaning effect on frequency bands
- Excluding the highest-variance component removes high-amplitude artifacts
- Core brain wave bands remain after cleaning
- Visual evaluation complements quantitative evaluation (SNR)

